# Labels, Titles, and Annotations

**Part I · Visualization** — Tutorial 09

Enhance visualizations with text: `Label` dataclasses, title overlays, and
Markdown annotation panels with LaTeX math (rendered via KaTeX). You will learn
to:

- Attach labels to entities (local-frame positioning, per-entity anchors,
  screen-plane rotation).
- Style labels and configure global/per-kind label defaults.
- Update labels dynamically and use KaTeX math.
- Add a title overlay and a Markdown annotation panel.


## Setup


In [ ]:
from pytanga.geometry import Direction, Line, Plane, Point, Sphere
from pytanga.viz import (
    AnnotationStyle, Label, LabelStyle, PointStyle, TitleStyle, Visualizer,
)


## 1. `Label` dataclass and the `label=` shortcut

`viz.add(..., label="text")` auto-creates a `Label` attached to the entity and
returns the **entity id** (use `viz.get_label_ids(entity_id)` for the attached
label ids). A standalone `Label` has a `text`, a 3D `position`, and an optional
`parent_id`.


In [ ]:
viz = Visualizer(title="Labels — basics", add_default_axes=False, add_default_grid=False)

# Convenience shortcut (auto-creates an attached Label):
viz.add(Point(1, 2, 0), color="#ff4444", style=PointStyle(size=0.15), label="$P_1$")

# Standalone labels:
eid = viz.add(Sphere(Point(5, 0, 0), 1.0), opacity=0.4)
viz.add(Label(text="My Sphere", position=(0, 1.3, 0), parent_id=eid))
viz.add(Label(text="Origin", position=(0, 0, 0)))

viz.display_snapshot()


## 2. `LabelStyle` — local-frame positioning

`LabelStyle` controls `font_size`, `font_family`, `color`, `background`,
`offset_local` (3D offset in the entity's local frame, scaled by its size),
`offset_2d` (screen-space pixel offset), `align`, `along`, and `rotation`.


In [ ]:
viz = Visualizer(title="Labels — style", add_default_axes=False, add_default_grid=False)

viz.add(
    Point(0, 0, 0),
    color="#ffff00",
    label="Origin",
    label_style=LabelStyle(
        offset_local=(0.0, 1.1, 0.0),   # above the point
        font_size=18,
        color="#ffff00",
        background="rgba(0, 0, 0, 0.8)",
    ),
)
viz.add(
    Sphere(Point(3, 0, 0), 1.2),
    opacity=0.4,
    label="$S_1$",
    label_style=LabelStyle(offset_local=(0.0, 1.05, 0.0)),  # just above the surface
)
viz.display_snapshot()


## 3. Per-entity anchors — `along`

`along` parameterizes where along an entity's extent the label anchors (a
scalar or a tuple of fractions). For a `Line`, `along=0.5` is the midpoint
(default) and `along=1.0` is the end.


In [ ]:
viz = Visualizer(title="Labels — along", add_default_axes=False, add_default_grid=False)

viz.add(
    Line.from_points(Point(0, 0, 0), Point(4, 0, 0)),
    color="#44aaff",
    label="mid",
    label_style=LabelStyle(along=0.5),
)
viz.add(
    Line.from_points(Point(0, 2, 0), Point(4, 2, 0)),
    color="#44ff44",
    label="end",
    label_style=LabelStyle(along=1.0),
)
viz.display_snapshot()


## 4. Screen-plane rotation — `rotation`

`rotation` (degrees, clockwise) rotates the label about its anchor in the
screen plane — useful for tick labels so longer labels don't overlap.


In [ ]:
viz = Visualizer(title="Labels — rotation", add_default_axes=False, add_default_grid=False)

viz.add(Point(0, 0, 0), color="#ff4444", label="R (45°)", label_style=LabelStyle(rotation=45))
viz.add(Point(3, 0, 0), color="#44aaff", label="R (-30°)", label_style=LabelStyle(rotation=-30))
viz.display_snapshot()


## 5. Default label styling

Set global label defaults via `viz.styles.label_base` and per-kind overrides via
`viz.styles.label_kind` (priority: user `label_style` > per-kind > global).


In [ ]:
viz = Visualizer(title="Labels — defaults", add_default_axes=False, add_default_grid=False)

viz.styles.label_base.offset_local = (0.0, 1.1, 0.0)
viz.styles.label_base.align = (0.5, 1.0)
viz.styles.label_kind["Sphere"] = LabelStyle(offset_local=(0.0, 1.05, 0.0))

viz.add(Point(0, 0, 0), color="#ff4444", label="P")
viz.add(Sphere(Point(3, 0, 0), 1.2), opacity=0.4, label="S")
viz.display_snapshot()


## 6. Updating labels dynamically

`update_label()` changes text and/or style without repositioning; an empty
`text=""` removes the label.


In [ ]:
viz = Visualizer(title="Labels — update", add_default_axes=False, add_default_grid=False)

eid = viz.add(Point(0, 0, 0), color="#ff4444", label="before")
lid = viz.get_label_ids(eid)[0]
viz.update_label(lid, text="after", style=LabelStyle(color="#00ff00"))
# viz.update_label(lid, text="")   # remove the label

viz.display_snapshot()


## 7. KaTeX math in labels

Label text containing `$...$` (inline) or `$$...$$` (display) is rendered as
LaTeX math via KaTeX — in the live viewer and all HTML exports. Use a raw
string (`r"..."`) for LaTeX commands so backslashes are preserved.


In [ ]:
viz = Visualizer(title="Labels — KaTeX", add_default_axes=False, add_default_grid=False)

viz.add(Point(1, 2, 0), color="#ff4444", label=r"$\mathbf{P}_1$")
viz.add(Sphere(Point(0, 0, 0), 2.0), opacity=0.3, label="$S_1$")
viz.add(Point(0, 0, 0), color="#ffff00", label=r"Origin: $\vec{0}$")

viz.display_snapshot()


## 8. Title overlay and `TitleStyle`

The `title` parameter (or `set_title()`) shows a fixed-position heading at the
top of the viewport; `TitleStyle` controls `font_size`, `color`, `background`.


In [ ]:
viz = Visualizer(title="Tanga — Title & Annotation")
viz.add(Sphere(Point(0, 0, 0), 1.5), opacity=0.3)

viz.set_title("Sphere Visualization")
# viz.styles.annotation  # mutate the global AnnotationStyle default

viz.display_snapshot()


## 9. Annotation panel and `AnnotationStyle`

The `annotation` parameter (or `set_annotation()`) renders **Markdown** with
**LaTeX math** in a scrollable panel at the bottom of the viewport (using
`marked` + `KaTeX`). Escape LaTeX backslashes (`\\frac`) inside a normal Python
string. `set_annotation(None)` hides the panel.


In [ ]:
viz = Visualizer(
    title="Tanga — Annotation",
    annotation="## Step 1\n\nThe sphere is defined by: $S = o - \\frac{1}{2} r^2 \\infty$\n\nIn conformal GA:\n$$S \\cdot X = 0$$",
)
viz.add(Sphere(Point(0, 0, 0), 1.5), opacity=0.3, label="$S$")

# Live update:
viz.set_annotation("## Step 2\n\n$R = e^{-i\\theta/2}$")
# viz.set_annotation(None)  # hide

viz.display_snapshot()


## Visual Examples

A presentation-ready figure — labelled sphere + point + annotation — exported
via `export_snapshot()`.


In [ ]:
viz = Visualizer(
    title="Tanga — Labelled Figure",
    annotation="A labelled sphere $S_1$ with centre $O$.",
)
viz.add(Sphere(Point(0, 0, 0), 2.0), opacity=0.4, label="$S_1$")
viz.add(Point(0, 0, 0), color="#ffff00", label="$O$", label_style=LabelStyle(offset_local=(0, 1.1, 0)))
viz.add(Line(origin=Point(-3, 0, 0), direction=Direction(1, 0, 0)), color="#44ff44", label="x-axis", label_style=LabelStyle(along=0.5))

viz.export_snapshot("_output/09_labels.html", overwrite=True)
print("labelled figure exported")


## Summary

| Task | API |
|---|---|
| Attached label | `add(entity, label="text")` |
| Standalone label | `add(Label(text, position, parent_id=...))` |
| Style | `LabelStyle(font_size, color, offset_local, offset_2d, align, along, rotation)` |
| Per-entity anchor | `LabelStyle(along=0.5)` |
| Screen rotation | `LabelStyle(rotation=45)` |
| Defaults | `viz.styles.label_base` / `viz.styles.label_kind` |
| Update | `viz.update_label(id, text=..., style=...)` |
| KaTeX | `"$...$"` / `"$$...$$"` in label text (raw string for commands) |
| Title | `set_title(...)` + `TitleStyle` |
| Annotation | `set_annotation(markdown)` + `AnnotationStyle` |

**Next:** [10 — Interaction](../10_interaction/).
